# FMAifs Postprocessing (CPU Regrid + CF Output)

This notebook converts raw rollout trajectories into CF-compliant NetCDF files.

## Pipeline steps

- Load rollout trajectories from pickle files
- Regrid fields from native model grid to regular lat-lon grid
- Convert trajectories into xarray Dataset
- Save output as NetCDF (CF-1.8 compliant)

## Output

Each input trajectory produces one NetCDF file.


In [ ]:
print("FMAifs POSTPROCESS (CPU REGRID + CF)")

import pickle
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import xarray as xr
import earthkit.regrid as ekr

In [ ]:
# -----------------------------
# CONFIG
# -----------------------------

# Assumptions:
# - Input directory contains pickle files generated by rollout step
# - Each file contains a trajectory list:
#     [
#         {
#             "date": datetime,
#             "fields": {var: np.ndarray}
#         },
#         ...
#     ]
# - Output will be written as CF-compliant NetCDF files

input_path = Path("./HHO_AIFS_output")
base_output = Path("./HHO_AIFS_output/cf_output")
surface_root = base_output / "surface_hourly" / "inst"
atmos_root = base_output / "atmos_hourly" / "inst"

In [ ]:
# --------------------------------------------------
# VARIABLES
# --------------------------------------------------
surface_vars = [
    "10u",
    "10v",
    "2d",
    "2t",
    "msl",
    "skt",
    "sp",
    "tcw",
]

rename_sfc = {
    "10u": "u10",
    "10v": "v10",
    "2d": "d2m",
    "2t": "t2m"
}

atmos_vars = [
    "z",
    "t",
    "u",
    "v",
    "w",
    "q",
]

pressure_levels = [
    1000,
    925,
    850,
    700,
    600,
    500,
    400,
    300,
    250,
    200,
    150,
    100,
    50,
]


# --------------------------------------------------
# TARGET GRID
# --------------------------------------------------
target_grid = {"grid": (0.25, 0.25)}

latitude = np.linspace(90, -90, 721)

longitude = np.linspace(
    0,
    359.75,
    1440,
)


# --------------------------------------------------
# VARIABLE ATTRIBUTES
# --------------------------------------------------
var_attrs = {
    "t": {
        "long_name": "Temperature",
        "standard_name": "air_temperature",
        "units": "K",
    },
    "u": {
        "long_name": "U component of wind",
        "standard_name": "eastward_wind",
        "units": "m s**-1",
    },
    "v": {
        "long_name": "V component of wind",
        "standard_name": "northward_wind",
        "units": "m s**-1",
    },
    "w": {
        "long_name": "Vertical velocity",
        "standard_name": "lagrangian_tendency_of_air_pressure",
        "units": "Pa s**-1",
    },
    "q": {
        "long_name": "Specific humidity",
        "standard_name": "specific_humidity",
        "units": "kg kg**-1",
    },
    "z": {
        "long_name": "Geopotential",
        "standard_name": "geopotential",
        "units": "m**2 s**-2",
    },
    "10u": {
        "long_name": "10 metre U wind component",
        "standard_name": "eastward_wind",
        "units": "m s**-1",
    },
    "10v": {
        "long_name": "10 metre V wind component",
        "standard_name": "northward_wind",
        "units": "m s**-1",
    },
    "2t": {
        "long_name": "2 metre temperature",
        "standard_name": "air_temperature",
        "units": "K",
    },
    "2d": {
        "long_name": "2 metre dewpoint temperature",
        "standard_name": "dew_point_temperature",
        "units": "K",
    },
    "msl": {
        "long_name": "Mean sea level pressure",
        "standard_name": "air_pressure_at_mean_sea_level",
        "units": "Pa",
    },
    "skt": {
        "long_name": "Skin temperature",
        "standard_name": "surface_temperature",
        "units": "K",
    },
    "sp": {
        "long_name": "Surface pressure",
        "standard_name": "surface_air_pressure",
        "units": "Pa",
    },
    "tcw": {
        "long_name": "Total column water",
        "standard_name": "atmosphere_mass_content_of_water_vapor",
        "units": "kg m**-2",
    },
}


In [ ]:
# --------------------------------------------------
# REGRID FUNCTION
# --------------------------------------------------
def regrid_field(flat_field):

    vals = ekr.interpolate(
        flat_field,
        {"grid": "N320"},
        target_grid,
    )

    vals = vals[::-1, :]

    nlon = vals.shape[1]

    vals = np.roll(vals, nlon // 2, axis=1)

    return vals.astype(np.float32)


# --------------------------------------------------
# BUILD SURFACE DATASET
# --------------------------------------------------
def build_surface_dataset(traj):

    times = [s["date"] for s in traj]

    data_vars = {}

    for var in surface_vars:

        print(f"Processing surface variable: {var}")

        arrays = []

        for s in traj:

            flat = s["fields"][var][1]

            vals = regrid_field(flat)

            arrays.append(vals)

        arr = np.stack(arrays)

        data_vars[var] = (
            (
                "valid_time",
                "latitude",
                "longitude",
            ),
            arr,
        )

    ds = xr.Dataset(
        data_vars=data_vars,
        coords={
            "valid_time": np.array(times),
            "latitude": latitude,
            "longitude": longitude,
        },
        attrs={
            "Conventions": "CF-1.7",
            "institution": "ECMWF",
        },
    )

    for var in surface_vars:

        ds[var].attrs = var_attrs[var]

    ds["valid_time"].attrs = {
        "long_name": "time",
        "standard_name": "time",
    }

    ds["latitude"].attrs = {
        "units": "degrees_north",
        "standard_name": "latitude",
        "long_name": "latitude",
        "stored_direction": "decreasing",
    }

    ds["longitude"].attrs = {
        "units": "degrees_east",
        "standard_name": "longitude",
        "long_name": "longitude",
    }

    return ds


# --------------------------------------------------
# BUILD ATMOSPHERIC DATASET
# --------------------------------------------------
def build_atmos_dataset(traj):

    times = [s["date"] for s in traj]

    data_vars = {}

    for base_var in atmos_vars:

        print(f"Processing atmospheric variable: {base_var}")

        level_arrays = []

        for level in pressure_levels:

            field_name = f"{base_var}_{level}"

            print(f"  {field_name}")

            time_arrays = []

            for s in traj:

                flat = s["fields"][field_name][1]

                vals = regrid_field(flat)

                time_arrays.append(vals)

            level_arr = np.stack(time_arrays)

            level_arrays.append(level_arr)

        arr4d = np.stack(level_arrays)

        arr4d = np.transpose(
            arr4d,
            (1, 0, 2, 3),
        )

        data_vars[base_var] = (
            (
                "valid_time",
                "pressure_level",
                "latitude",
                "longitude",
            ),
            arr4d.astype(np.float32),
        )

    ds = xr.Dataset(
        data_vars=data_vars,
        coords={
            "valid_time": np.array(times),
            "pressure_level": np.array(
                pressure_levels,
                dtype=np.float64,
            ),
            "latitude": latitude,
            "longitude": longitude,
        },
        attrs={
            "Conventions": "CF-1.7",
            "institution": "ECMWF",
        },
    )

    for var in atmos_vars:

        ds[var].attrs = var_attrs[var]

    ds["valid_time"].attrs = {
        "long_name": "time",
        "standard_name": "time",
    }

    ds["pressure_level"].attrs = {
        "long_name": "pressure",
        "units": "hPa",
        "positive": "down",
        "stored_direction": "decreasing",
        "standard_name": "air_pressure",
    }

    ds["latitude"].attrs = {
        "units": "degrees_north",
        "standard_name": "latitude",
        "long_name": "latitude",
        "stored_direction": "decreasing",
    }

    ds["longitude"].attrs = {
        "units": "degrees_east",
        "standard_name": "longitude",
        "long_name": "longitude",
    }

    return ds


## Main Processing Loop

This section:

- Iterates over all rollout files
- Converts each trajectory into a CF-compliant dataset
- Writes NetCDF output files


In [ ]:
start_date = datetime(2024, 4, 30, 12)
end_date = datetime(2024, 5, 31, 0)
current_date = start_date

while current_date <= end_date:

    timestamp = current_date.strftime("%Y%m%d_%H")

    infile = input_path / f"HHO_aifs_rollout_{timestamp}.pkl"

    print("")
    print("====================================")
    print(f"Processing {infile}")
    print("====================================")

    if not infile.exists():

        print("Missing file")

        current_date += timedelta(hours=6)

        continue

    with open(infile, "rb") as f:

        trajectory = pickle.load(f)

    init_time = trajectory[0]["date"]

    surface_ds_all = build_surface_dataset(
        trajectory
    ).rename(rename_sfc)

    atmos_ds_all = build_atmos_dataset(
        trajectory
    )

    for itime in range(surface_ds_all.sizes["valid_time"]):

        valid_time = surface_ds_all["valid_time"].values[itime]

        valid_time_dt = np.datetime64(valid_time).astype("datetime64[s]").astype(datetime)

        year_dir = f"Y{init_time.year:04d}"
        month_dir = f"M{init_time.month:02d}"
        day_dir = f"D{init_time.day:02d}"

        surface_dir = (
            surface_root /
            year_dir /
            month_dir /
            day_dir
        )

        atmos_dir = (
            atmos_root /
            year_dir /
            month_dir /
            day_dir
        )

        surface_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        atmos_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        out_time = valid_time_dt.strftime("%Y%m%d_%Hz")

        surface_file = (
            surface_dir /
            f"era5_surface-inst_allvar_{out_time}.nc"
        )

        atmos_file = (
            atmos_dir /
            f"era5_atmos-inst_allvar_{out_time}.nc"
        )

        surface_ds = surface_ds_all.isel(valid_time=[itime])

        atmos_ds = atmos_ds_all.isel(valid_time=[itime])

        surface_encoding = {
            var: {"dtype": "float32"}
            for var in surface_ds.data_vars
        }

        atmos_encoding = {
            var: {"dtype": "float32"}
            for var in atmos_ds.data_vars
        }

        print("")
        print("Writing surface file")
        print(surface_file)

        surface_ds.to_netcdf(
            surface_file,
            encoding=surface_encoding,
        )

        print("")
        print("Writing atmospheric file")
        print(atmos_file)

        atmos_ds.to_netcdf(
            atmos_file,
            encoding=atmos_encoding,
        )

    current_date += timedelta(hours=12)


print("")
print("Done")
